In [2]:
import numpy as np
import torch
import torchmetrics
import torchvision
from sklearn.datasets import load_sample_images
from torch.ao.nn.quantized import BatchNorm2d
from torch.nn import CrossEntropyLoss

sample_images = np.stack(load_sample_images()["images"])
sample_images = torch.tensor(sample_images, dtype=torch.float32) / 255

In [4]:
sample_images.shape

torch.Size([2, 427, 640, 3])

In [7]:
sample_images_permuted = sample_images.permute(0, 3, 1, 2)
sample_images_permuted.shape

torch.Size([2, 3, 427, 640])

In [6]:
import torchvision
import torchvision.transforms.v2 as T
cropped_images = T.CenterCrop((70, 120))(sample_images_permuted)
cropped_images.shape

torch.Size([2, 3, 70, 120])

In [7]:
import torch.nn as nn

torch.manual_seed(42)
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7)
fmaps = conv_layer(cropped_images)

In [8]:
fmaps.shape

torch.Size([2, 32, 64, 114])

In [9]:
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7, padding="same")
fmaps = conv_layer(cropped_images)
fmaps.shape

torch.Size([2, 32, 70, 120])

In [10]:
conv_layer.weight.shape

torch.Size([32, 3, 7, 7])

In [11]:
conv_layer.bias.shape

torch.Size([32])

In [12]:
max_pool = nn.MaxPool2d(kernel_size=2)
avg_pool = nn.AvgPool2d(kernel_size=2)

In [13]:
import torch.functional as F

class DepthPool(nn.Module):
    def __init__(self, kernel_size, stride=None, padding=0):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride if stride is not None else kernel_size
        self.padding = padding


    def forward(self, inputs):
        batch, channels, height, width = inputs.shape
        Z = inputs.view(batch, channels, height * width)  # merge spatial dims
        Z = Z.permute(0, 2, 1)  # switch spatial and channels dims
        Z = F.max_pool1d(Z, kernel_size=self.kernel_size, stride=self.stride,
                         padding=self.padding)  # compute max pool
        Z = Z.permute(0, 2, 1)  # switch back spatial and channels dims
        return Z.view(batch, -1, height, width)  # unmerge spatial dims

In [14]:
global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1)
output = global_avg_pool(cropped_images)

In [15]:
output = cropped_images.mean(dim=(2, 3), keepdim=True)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [17]:
from functools import partial

DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same")

model = nn.Sequential(
    DefaultConv2d(in_channels=1, out_channels=64, kernel_size=7), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(64, 128), nn.ReLU(),
    DefaultConv2d(128, 128), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(128, 256), nn.ReLU(),
    DefaultConv2d(256, 256), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(2304, 128), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(64, 47)  # EMNIST balanced has 47 classes
).to(device)

In [19]:
import torchmetrics
from torch.utils.data import DataLoader
from Neural_Networks_Deep_Learning.CIFAR10.utils import train

transform = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])
train_and_valid_data = torchvision.datasets.EMNIST(
    root="datasets", split="balanced", train=True, transform=transform)

test_data = torchvision.datasets.EMNIST(
    root="datasets", split="balanced", train=False, transform=transform)

train_data, valid_data = torch.utils.data.random_split(train_and_valid_data, [107800, 5000])

train_loader = DataLoader(train_data,  batch_size=32, num_workers=4, pin_memory=True, shuffle=True)
valid_loader = DataLoader(valid_data,  batch_size=32, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_data,   batch_size=32, num_workers=4, pin_memory=True)

n_epochs = 100
optimizer = torch.optim.AdamW(model.parameters())
criterion = nn.CrossEntropyLoss()
metric = torchmetrics.Accuracy(task="multiclass", num_classes=47).to(device)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, max_lr=1e-2, total_steps=len(train_loader)*n_epochs)
n_iter_no_improvements = 3
train(model, optimizer, criterion, train_loader, valid_loader, metric, n_epochs, n_iter_no_improvements, scheduler=scheduler)

Epoch: 1/100, Loss: 1.8130, Val Score: 0.8094


KeyboardInterrupt: 

In [20]:
from Neural_Networks_Deep_Learning.CIFAR10.utils import evaluate

evaluate(model, test_loader, metric)

0.8053723573684692

In [21]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.depthwise_conv = nn.Conv2d(
            in_channels, in_channels, kernel_size, stride=stride,
            padding=padding, groups=in_channels)
        self.pointwise_conv = nn.Conv2d(
            in_channels, out_channels, kernel_size=1, stride=1, padding=0)

    def forward(self, inputs):
        return self.pointwise_conv(self.depthwise_conv(inputs))

In [ ]:
class ResidualUnit(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        DefaultConv2d = partial(
            nn.Conv2d, kernel_size=3, stride=1, padding=1, bias=False)
        self.main_layers = nn.Sequential(
            DefaultConv2d(in_channels, out_channels, stride=stride),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            DefaultConv2d(out_channels, out_channels),
            nn.BatchNorm2d(out_channels),
        )
        if stride > 1:
            self.skip_connection = nn.Sequential(
                DefaultConv2d(in_channels, out_channels, kernel_size=1,
                              stride=stride, padding=0),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.skip_connection = nn.Identity()

    def forward(self, inputs):
        return F.relu(self.main_layers(inputs) + self.skip_connection(inputs))

In [ ]:
class ResNet34(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2,
                      padding=3, bias=False),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        ]
        prev_filters = 64
        for filters in [64] * 3 + [128] * 4 + [256] * 6 + [512] * 3:
            stride = 1 if filters == prev_filters else 2
            layers.append(ResidualUnit(prev_filters, filters, stride=stride))
            prev_filters = filters
        layers += [
            nn.AdaptiveAvgPool2d(output_size=1),
            nn.Flatten(),
            nn.LazyLinear(10),
        ]
        self.resnet = nn.Sequential(*layers)

    def forward(self, inputs):
        return self.resnet(inputs)

In [5]:
weights = torchvision.models.ConvNeXt_Base_Weights.IMAGENET1K_V1
model = torchvision.models.convnext_base(weights=weights).to(device)

In [8]:
transforms = weights.transforms()
preprocessed_images = transforms(sample_images_permuted)

In [9]:
model.eval()
with torch.no_grad():
    y_logits = model(preprocessed_images.to(device))

In [11]:
y_pred = torch.argmax(y_logits, dim=1)
y_pred

tensor([698, 985], device='cuda:0')

In [13]:
class_names = weights.meta["categories"]
[class_names[class_id] for class_id in y_pred]

['palace', 'daisy']

In [15]:
y_top3_logits, y_top3_class_ids = y_logits.topk(k=3, dim=1)
[[class_names[class_id] for class_id in top3] for top3 in y_top3_class_ids]

[['palace', 'monastery', 'lakeside'], ['daisy', 'pot', 'ant']]

In [16]:
y_top3_logits.softmax(dim=1)

tensor([[0.8618, 0.1185, 0.0197],
        [0.8106, 0.0964, 0.0930]], device='cuda:0')